In [ ]:
from Modules.visual_analyser import VisualAnalyzer
from Modules.data_preprocessing import DataPreprocessing
from Modules.image_analyzer import ImageAnalyzer
import os

In [ ]:
# recupération des données
dp = DataPreprocessing()
source_path = "."
df_path = os.path
df = dp.read_data(os.path.join(source_path, "post_rehydrated.pickle"), format_="pickle")
dp.parse_dates()
# filtrage des données (On ne garde que les commentaires et quotes pas les originaux)
df = df[df["join_post_post_type"] != "original"]

## Statistiques descriptives : Images avec imageId

### **Quelles sont les images les plus publiées et les comptes qui les publient ?**

In [ ]:
image_analyzer = ImageAnalyzer(df=df)

In [ ]:
# seuil : 10
threshold = 10
image_popularity = image_analyzer.image_popularity()
image_popularity = image_popularity[image_popularity["n_posts"] > threshold]
image_popularity

In [ ]:
image_analyzer.display_specific_imgs(
    badges=image_popularity["n_posts"].tolist(),
    image_names=image_popularity["image_id"].tolist(),
    from_internet=False,
    ncols=4,
)

In [ ]:
publisher = image_analyzer.display_accounts_by_img_published(
    image_ids=image_popularity["image_id"].tolist()
)

In [ ]:
publisher.loc[
    (publisher["n_posts"] > 0)
    & (publisher.index.get_level_values(0) == "EWxYgVmXkAEmDEY.jpg")
]

In [ ]:
publisher.loc[
    (publisher["n_posts"] > 0)
    & (publisher.index.get_level_values(0) == "GkFfX5sXAAA3J48.jpg")
]

In [ ]:
publisher.loc[
    (publisher["n_posts"] > 0)
    & (publisher.index.get_level_values(0) == "Gk-48BiXIAAb3de.jpg")
]

Interessant, cette image a été publiée par plusieurs comptent distincts.

In [ ]:
publisher.loc[
    (publisher["n_posts"] > 0)
    & (publisher.index.get_level_values(0) == "Gk-48BiXIAAb3de.jpg")
]

In [ ]:
image_analyzer.display_specific_imgs(
    image_names=image_popularity["image_id"]
    .loc[(image_popularity["n_posts"] <= 25) & (image_popularity["n_posts"] >= 15)]
    .tolist(),
    ncols=2,
    from_internet=True,
)

Certaines images les plus viralisées (avec les textes) ont attraits avec le délévéloppement informatiques. Alors pourquoi Nicolas les publient également ?

In [ ]:
test_images = (
    image_popularity["image_id"]
    .loc[(image_popularity["n_posts"] <= 25) & (image_popularity["n_posts"] >= 15)]
    .tolist()[:4]
)

In [ ]:
image_analyzer.display_specific_imgs(
    image_names=test_images, ncols=2, from_internet=True
)

In [ ]:
from PIL import Image
import imagehash


def compute_image_hash(img_list, image_loader):
    result = {}
    for i in range(1, len(img_list)):
        img_prev = image_loader(img_list[i - 1])
        img_prev.resize(10, 10)
        img_prev.convert("L")
        img_curr = image_loader(img_list[i])
        img_curr.resize(10, 10)
        img_curr.convert("L")
        pHash_prev = imagehash.phash(img_prev)
        pHash_current = imagehash.phash(img_curr)

        hamming = pHash_current - pHash_prev

        key = f"{img_list[i-1]} vs {img_list[i]}"
        result[key] = hamming

    return result

In [ ]:
image_loader = image_analyzer.load_twitter_image
compute_image_hash(test_images, image_analyzer.load_twitter_image)

In [ ]:
image_intensity = image_analyzer.image_intensity()
image_intensity

In [ ]:
image_intensity = image_intensity[image_intensity["posts_per_account"] > 5]

In [ ]:
image_analyzer.display_specific_imgs(
    badges=image_intensity["posts_per_account"].tolist(),
    image_names=image_intensity["image_id"].tolist(),
    from_internet=False,
    ncols=3,
)

In [ ]:
for img, df_img in image_analyzer.df.groupby("image_id"):
    print(img, len(df_img))

In [ ]:
image_burst = image_analyzer.image_bursts(timeframe="35min")

In [ ]:
image_burst.sort_values(["image_id"])

In [ ]:
features_images = image_analyzer.build_features()

In [ ]:
features_images

image_id | account_id | timestamp
------------------------------------
A        | u12        | 2024-01-01 08:01
A        | u47        | 2024-01-01 08:03
A        | u71        | 2024-01-01 08:10
B        | u05        | 2024-01-01 15:22
B        | u33        | 2024-01-02 10:40


In [ ]:
def image_loader(img):
    img_path = None
    images_dir = [
        os.path.join(source_path, os.path.join(d, d)) for d in ["img", "source_img"]
    ]
    for d in images_dir:
        candidate = os.path.join(source_path, d, d, img)
        if os.path.exists(candidate):
            img_path = candidate
            break

    if img_path is None:
        raise
    img = Image.open(img_path).convert("RGB")
    return img

In [ ]:
image_dir = [
    os.path.join(source_path, os.path.join(d, d)) for d in ["img", "source_img"]
]
image_dir

In [ ]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"

# Charger le modèle CLIP et sa fonction de preprocess
model, preprocess_clip = clip.load("ViT-B/32", device=device)

In [ ]:
from Modules.visual_analyser import VisualAnalyzer

v_anal = VisualAnalyzer(df=df, clip_model=model, preprocess=preprocess_clip)

In [ ]:
v_anal.image_loader = image_loader

In [ ]:
v_anal.image_loader

In [ ]:
# suppression des doublons
# Calcul pHash
v_anal.compute_image_hash()

# Construire le graphe de duplications avec threshold=5
v_anal.build_dup_graph(threshold=5)

In [ ]:
import networkx as nx

In [ ]:
# Garder une seule image par composant connecté
dup_clusters = list(nx.connected_components(v_anal.dup_graph))
print(len(dup_clusters), len(v_anal.df["image_id"].unique()))
images_to_keep = set(next(iter(c)) for c in dup_clusters)
v_anal.df = v_anal.df[v_anal.df["image_id"].isin(images_to_keep)].reset_index(drop=True)
print(
    f"{len(v_anal.df['image_id'].unique())} images restent après suppression des doublons"
)

Aucun doublon

In [ ]:
v_anal.compute_clip_embeddings()

In [ ]:
v_anal.image_embeddings

In [ ]:
import numpy as np

In [ ]:
emb_matrix = np.array([emb.numpy() for emb in v_anal.image_embeddings.values()])
img_ids = np.array(list(v_anal.image_embeddings.keys()))

# Sauvegarde compressée
np.savez_compressed("image_embeddings.npz", embeddings=emb_matrix, ids=img_ids)
print("Embeddings sauvegardés dans image_embeddings.npz")

In [ ]:
data = np.load("image_embeddings.npz")
emb_matrix = data["embeddings"]
img_ids = data["ids"]

In [ ]:
img_ids

In [ ]:
import pandas as pd

In [ ]:
df_images = pd.DataFrame(emb_matrix)

In [ ]:
df_images["image_id"] = img_ids

In [ ]:
df_images["image_id"]

In [ ]:
from Modules.umap import UmapDimensionReducer

In [ ]:
ump = UmapDimensionReducer(n_neighbours=15, n_components=2, data=df_images)

In [ ]:
ump.reduce()
print(ump.reducer)

In [ ]:
ump.fit_map()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
import umap
import hdbscan

# ============================================================
# INPUT
# embeddings : np.ndarray de shape (N, 512)
# image_ids  : list[str] de longueur N (même ordre)
# ============================================================

assert emb_matrix.shape[0] == len(img_ids), "Mismatch embeddings / image_ids"

# ============================================================
# DataFrame avec image_id comme rownames
# ============================================================

df = pd.DataFrame(emb_matrix, index=img_ids)

# ============================================================
# Normalisation L2 (très important pour CLIP)
# ============================================================

X = normalize(df.values, norm="l2")

# ============================================================
# UMAP (réduction de dimension pour clustering)
# ============================================================

umap_model = umap.UMAP(
    n_neighbors=30,  # 15–50 selon la densité
    n_components=10,  # dimensions finales pour clustering
    min_dist=0.0,  # clusters plus compacts
    metric="cosine",  # cohérent avec CLIP
    random_state=42,
)

X_umap = umap_model.fit_transform(X)

# ============================================================
# HDBSCAN (clustering)
# ============================================================

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,
    metric="euclidean",  # UMAP sort en espace euclidien
    cluster_selection_method="eom",
)

labels = clusterer.fit_predict(X_umap)

# ============================================================
# Résultats finaux
# ============================================================

results = pd.DataFrame({"image_id": df.index, "cluster": labels})

print(results.head(10))

In [ ]:
print(results.head(10))

In [ ]:
images_cluster_id = 2
img_clusters = results.loc[results["cluster"] == images_cluster_id, "image_id"].tolist()

In [ ]:
import random

random.shuffle(img_clusters)

## y a trop de doublons

In [ ]:
len(img_clusters)

In [ ]:
image_analyzer.display_specific_imgs(image_names=img_clusters[:100])

In [ ]:
# H1
import networkx as nx

G_time = nx.Graph()
for cluster_id in df_clean["cluster"].unique():
    cluster_data = df_clean[df_clean["cluster"] == cluster_id]
    # On ne garde que les comptes publiant dans Δt < τ
    sorted_posts = cluster_data.sort_values("post_created_at")
    for i, row_i in sorted_posts.iterrows():
        for j, row_j in sorted_posts.iterrows():
            if i >= j:
                continue
            delta_t = (
                row_j["post_created_at"] - row_i["post_created_at"]
            ).total_seconds()
            if delta_t < tau_seconds:
                G_time.add_edge(
                    row_i["account_id"], row_j["account_id"], cluster=cluster_id
                )

In [ ]:
# h2 : ponderation
G_engagement = nx.Graph()
for cluster_id in df_clean["cluster"].unique():
    cluster_data = df_clean[df_clean["cluster"] == cluster_id]
    high_engagement = cluster_data[cluster_data["engagement"] > engagement_threshold]
    for i, row_i in high_engagement.iterrows():
        for j, row_j in high_engagement.iterrows():
            if i >= j:
                continue
            G_engagement.add_edge(
                row_i["account_id"], row_j["account_id"], cluster=cluster_id
            )

In [ ]:
# H3
G_network = nx.Graph()
# On suppose que tu as une table "reposts" avec (source_account, target_account, cluster_id)
for _, row in df_reposts.iterrows():
    if row["cluster_id"] in df_clean["cluster"].unique():
        G_network.add_edge(
            row["source_account"], row["target_account"], cluster=row["cluster_id"]
        )